# Card Database and Starter-Deck EDA

**Purpose.** Audit the official English card catalogue and starter deck
before changing either policy or deck construction.

**Decision question.** Which card-pool and deck properties constrain our
first agent experiments?

This is a simulation competition: EDA means catalogue, deck, state-space,
and episode analysisâ€”not train/test target exploration. Run with the
`pokemon-tcg-ai-battle` competition data attached. A local `data/raw`
fallback is supported.

## 1. Configuration

The input resolver avoids account-specific paths. We keep plots consistent
and separate card identity from move-level records because one card can
occupy several catalogue rows.

In [ ]:
from collections import Counter
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_FILENAME = "EN_Card_Data.csv"
sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 50)

def find_file(filename: str) -> Path:
    root = Path("/kaggle/input")
    matches = sorted(root.rglob(filename)) if root.exists() else []
    if matches:
        return matches[0]
    local = Path("../data/raw") / filename
    if local.exists():
        return local
    raise FileNotFoundError(
        f"Attach competition data or place {filename} in data/raw/."
    )

card_path = find_file(DATA_FILENAME)
print(f"Catalogue: {card_path}")

## 2. Schema and representation audit

We distinguish catalogue rows, unique card IDs, and cardâ€“move records.
Structural missingness is expected: Energy and Trainer cards do not have
PokÃ©mon HP or evolution fields.

In [ ]:
raw = pd.read_csv(card_path)
cards = raw.rename(columns=lambda x: re.sub(
    r"[^a-z0-9]+", "_", x.strip().lower()
).strip("_"))

summary = pd.Series({
    "catalogue_rows": len(cards),
    "columns": cards.shape[1],
    "unique_card_ids": cards.card_id.nunique(),
    "exact_duplicate_rows": cards.duplicated().sum(),
})
display(summary.to_frame("value"))
display(cards.head())

quality = pd.DataFrame({
    "dtype": cards.dtypes.astype(str),
    "missing": cards.isna().sum(),
    "missing_pct": cards.isna().mean().mul(100).round(2),
    "unique": cards.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)
display(quality)

**Interpretation.** Do not globally impute the catalogue. Parse and analyze
fields within relevant card categories, and use `Card ID` as the stable
identity key. Names and move rows are not unique enough for policy logic.

## 3. Card-pool composition

Collapse identity fields to one record per card for composition plots.
Attacks remain a separate one-to-many table for later action scoring.

In [ ]:
category_col = "stage_pok_mon_type_energy_and_trainer"
identity = [
    "card_id", "card_name", "expansion", category_col, "rule", "category",
    "previous_stage", "hp", "type", "weakness", "resistance_type", "retreat",
]
identity = [column for column in identity if column in cards]
unique_cards = cards[identity].groupby("card_id", as_index=False).first()

counts = unique_cards[category_col].fillna("Missing").value_counts().head(15)
ax = counts.sort_values().plot.barh(
    figsize=(10, 6), color=sns.color_palette("viridis", len(counts))
)
ax.set(title="Largest card categories", xlabel="Unique cards", ylabel="Category")
plt.tight_layout()
plt.show()

unique_cards["hp_numeric"] = pd.to_numeric(unique_cards.hp, errors="coerce")
unique_cards["retreat_numeric"] = pd.to_numeric(unique_cards.retreat, errors="coerce")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(unique_cards.hp_numeric.dropna(), bins=30, ax=axes[0])
axes[0].set_title("Printed HP distribution")
sns.countplot(x=unique_cards.retreat_numeric, ax=axes[1], color="#2a788e")
axes[1].set_title("Retreat-cost distribution")
plt.tight_layout()
plt.show()

**Interpretation.** Catalogue frequency is descriptive, not a deck recipe.
Deck strength depends on legal counts, evolution support, energy curve,
interactions, and whether the policy can sequence them correctly.

## 4. Starter-deck audit

Basic Energy can exceed the ordinary four-copy heuristic. We flag other
large counts for review rather than claiming that a simplified check is the
official legality test. The simulator start result remains authoritative.

In [ ]:
def find_deck() -> Path:
    candidates = [Path("../agent/deck.csv"), Path("agent/deck.csv")]
    candidates += sorted(Path("/kaggle/input").rglob("sample_submission/deck.csv"))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find a repository or official starter deck.")

deck_path = find_deck()
deck = [int(x) for x in deck_path.read_text().splitlines() if x.strip()]
deck_counts = pd.Series(Counter(deck), name="copies").rename_axis("card_id").reset_index()
columns = ["card_id", "card_name", category_col, "hp", "retreat"]
deck_view = deck_counts.merge(unique_cards[columns], on="card_id", how="left")
deck_view = deck_view.sort_values(["copies", "card_id"], ascending=[False, True])

print(f"Deck: {deck_path}")
print(f"Cards: {len(deck)}; unique IDs: {len(deck_counts)}")
display(deck_view)
assert len(deck) == 60, "Submission decks require exactly 60 cards."
assert deck_view.card_name.notna().all(), "All deck IDs must exist in the catalogue."

basic_energy = deck_view[category_col].fillna("").str.contains(
    "Basic Energy", case=False, regex=False
)
print("Non-Basic-Energy entries above four copies (manual review):")
display(deck_view[(deck_view.copies > 4) & ~basic_energy])

## 5. Findings and next experiment

- Preserve card ID as the stable join key.
- Separate card identity from attacks before creating value tables.
- Interpret missingness by category.
- Use simulator initialization as the final executable deck check.
- Next, run the deterministic baseline and capture context telemetry.

**Limitation.** Printed card data does not reveal strategic synergy or
opponent prevalence. Those require controlled episode evidence.